# Blender One-Click Render — ChatGPT Work\n\nVersión corregida para **Colab + NVIDIA T4 + EEVEE/Vulkan**.\n\nUso normal: selecciona **GPU T4**, pulsa **Ejecutar todo**, autoriza Drive y espera. El cuaderno hace un smoke test antes del render completo para no perder tiempo si el backend no funciona.\n

In [ ]:
# @title Blender One-Click Render (GPU T4 / EEVEE Vulkan)
# Ejecuta TODO el cuaderno. Solo autoriza Google Drive cuando Colab lo pida.

PROJECT_SLUG = "blender-neon-core-demo"
REPO_URL = "https://github.com/izquierdodavid528-code/Chat-gpt-work-.git"
REPO_DIR = "/content/Chat-gpt-work-"
BLENDER_VERSION = "4.5.14"
BLENDER_DIR = f"/content/blender-{BLENDER_VERSION}-linux-x64"
BLENDER_BIN = f"{BLENDER_DIR}/blender"

import os, json, time, shutil, pathlib, subprocess, threading, re
from datetime import datetime, timezone

def section(s):
    print("\n" + "="*72 + "\n" + s + "\n" + "="*72)

def sh(cmd, check=True, capture=False):
    print("$", cmd)
    p = subprocess.run(["bash","-lc",cmd], text=True,
                       capture_output=capture)
    if capture:
        if p.stdout: print(p.stdout)
        if p.stderr: print(p.stderr)
    if check and p.returncode != 0:
        raise RuntimeError(f"Comando fallo ({p.returncode}): {cmd}")
    return p

section("1/9 - Comprobando GPU NVIDIA")
gpu = sh("nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv,noheader",
         check=False, capture=True)
if gpu.returncode != 0 or not gpu.stdout.strip():
    raise RuntimeError(
        "No hay GPU NVIDIA activa. En Colab: Entorno de ejecucion > "
        "Cambiar tipo de entorno de ejecucion > GPU T4 > Guardar. "
        "Despues vuelve a Ejecutar todo."
    )
print("GPU OK:", gpu.stdout.strip())

section("2/9 - Montando Google Drive")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
DRIVE_ROOT = pathlib.Path("/content/drive/MyDrive/Remotion Projects")
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

section("3/9 - Descargando workspace")
if pathlib.Path(REPO_DIR).exists():
    shutil.rmtree(REPO_DIR)
sh(f'git clone --depth 1 "{REPO_URL}" "{REPO_DIR}"')
os.chdir(REPO_DIR)

section("4/9 - Instalando Blender 4.5 LTS y Vulkan")
sh("apt-get -qq update")
sh("apt-get -qq install -y ffmpeg libegl1 libgl1 libx11-6 libxi6 libxrender1 libxfixes3 libxxf86vm1 libxkbcommon0 libvulkan1 vulkan-tools")
if not pathlib.Path(BLENDER_BIN).exists():
    url=f"https://download.blender.org/release/Blender4.5/blender-{BLENDER_VERSION}-linux-x64.tar.xz"
    sh(f'wget -q --show-progress "{url}" -O /content/blender.tar.xz')
    sh("tar -xf /content/blender.tar.xz -C /content")
sh(f'"{BLENDER_BIN}" --version | head -n 3')
vulkan = sh("vulkaninfo --summary 2>/dev/null | head -n 80", check=False, capture=True)
if "Tesla T4" not in (vulkan.stdout or "") and "NVIDIA" not in (vulkan.stdout or ""):
    print("AVISO: vulkaninfo no identifico claramente la T4, pero se intentara Blender Vulkan.")

section("5/9 - Cargando proyecto")
project_dir = pathlib.Path(REPO_DIR)/"projects"/PROJECT_SLUG
cfg = json.load(open(project_dir/"project.config.json", encoding="utf-8"))
scene_script = project_dir/cfg.get("sceneScript","scene.py")
out_dir = project_dir/"out"
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)
print(json.dumps(cfg, indent=2, ensure_ascii=False))

# Crear copias temporales del script SIN el poster inicial.
src = scene_script.read_text(encoding="utf-8")
src_no_poster = src.replace(
    'bpy.ops.render.render(write_still=True)',
    'print("COLAB: poster omitido para acelerar la prueba")',
    1
)
full_scene = pathlib.Path("/content/colab_scene_full.py")
full_scene.write_text(src_no_poster, encoding="utf-8")

# Smoke test: 25% de resolucion y solo 1 frame.
smoke_src = src_no_poster.replace(
    'scene.frame_set(1)\nbpy.ops.render.render(animation=True)',
    'scene.render.resolution_percentage = 25\nscene.frame_end = 1\nscene.frame_set(1)\nbpy.ops.render.render(animation=True)',
    1
)
smoke_scene = pathlib.Path("/content/colab_scene_smoke.py")
smoke_scene.write_text(smoke_src, encoding="utf-8")

env = os.environ.copy()
env.pop("LIBGL_ALWAYS_SOFTWARE", None)
env.pop("MESA_LOADER_DRIVER_OVERRIDE", None)

def run_blender(scene_file, progress_file, log_file, timeout=None, show_progress=True):
    env2 = env.copy()
    env2["BLENDER_PROJECT_SLUG"] = PROJECT_SLUG
    env2["BLENDER_PROGRESS_FILE"] = str(progress_file)
    cmd = [
        BLENDER_BIN,
        "--gpu-backend", "vulkan",
        "-b",
        "--python", str(pathlib.Path(REPO_DIR)/"scripts/blender/render-with-progress.py"),
        "--", str(scene_file)
    ]
    print("Comando:", " ".join(cmd))
    start=time.time()
    last=None
    with open(log_file,"w",encoding="utf-8") as log:
        p=subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env2, text=True)
        while p.poll() is None:
            if timeout and time.time()-start > timeout:
                p.terminate()
                try: p.wait(15)
                except: p.kill()
                raise RuntimeError(
                    f"Smoke test excedio {timeout}s. No conviene seguir con este backend."
                )
            if show_progress and progress_file.exists():
                try:
                    d=json.load(open(progress_file,encoding="utf-8"))
                    stamp=(d.get("percent"),d.get("current_frame"),d.get("eta_seconds"))
                    if stamp!=last:
                        print(f"Progreso: {d.get('percent')}% | frame {d.get('current_frame')}/{d.get('frame_end')} | ETA {d.get('eta_seconds')} s")
                        last=stamp
                except Exception:
                    pass
            time.sleep(3)
    elapsed=time.time()-start
    if p.returncode != 0:
        tail="\n".join(open(log_file,errors="ignore").read().splitlines()[-120:])
        print(tail)
        raise RuntimeError("Blender fallo; revisa el log mostrado arriba.")
    return elapsed

section("6/9 - Smoke test GPU: 1 frame a 25%")
smoke_progress=out_dir/"smoke-progress.json"
smoke_log=out_dir/"smoke-render.log"
smoke_start=time.time()
smoke_elapsed=run_blender(smoke_scene, smoke_progress, smoke_log, timeout=120, show_progress=False)
print(f"Smoke test OK: {smoke_elapsed:.1f}s")

section("7/9 - Render completo en EEVEE + Vulkan")
progress_file=out_dir/"render-progress.json"
log_file=out_dir/"colab-render.log"
gpu_log=out_dir/"gpu-usage.csv"

stop_poll=threading.Event()
def poll_gpu():
    with open(gpu_log,"w",encoding="utf-8") as f:
        f.write("timestamp,gpu_util_percent,memory_used_mb\n")
        while not stop_poll.is_set():
            p=subprocess.run(
                ["bash","-lc","nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader,nounits"],
                text=True,capture_output=True
            )
            if p.returncode==0 and p.stdout.strip():
                vals=p.stdout.strip().splitlines()[0]
                f.write(f"{datetime.now(timezone.utc).isoformat()},{vals}\n")
                f.flush()
            stop_poll.wait(5)

t=threading.Thread(target=poll_gpu,daemon=True)
t.start()
try:
    elapsed=run_blender(full_scene, progress_file, log_file, timeout=None, show_progress=True)
finally:
    stop_poll.set()
    t.join(timeout=10)

print(f"Render completo: {elapsed/60:.2f} min ({elapsed:.1f}s)")

section("8/9 - Normalizando salida y creando benchmark")
expected_video=cfg.get("videoFile")
expected_path=pathlib.Path(REPO_DIR)/expected_video if expected_video else None
candidates=[]
for pat in ("*.mp4","*.mkv","*.mov","*.webm","*.avi"):
    candidates.extend(out_dir.glob(pat))

if expected_path and not expected_path.exists() and candidates:
    src_video=max(candidates,key=lambda p:p.stat().st_size)
    tmp=out_dir/(expected_path.name+".tmp.mp4")
    print("Normalizando:",src_video.name,"->",expected_path.name)
    q=subprocess.run(["ffmpeg","-y","-i",str(src_video),"-c","copy",str(tmp)],
                     stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    if q.returncode!=0:
        subprocess.run(["ffmpeg","-y","-i",str(src_video),"-c:v","libx264","-preset","veryfast","-crf","18","-an",str(tmp)],
                       check=True,stdout=subprocess.DEVNULL,stderr=subprocess.DEVNULL)
    tmp.replace(expected_path)

utils=[]
if gpu_log.exists():
    for line in gpu_log.read_text(encoding="utf-8").splitlines()[1:]:
        try: utils.append(float(line.split(",")[1].strip()))
        except: pass

summary={
    "project":PROJECT_SLUG,
    "blender_version":BLENDER_VERSION,
    "backend":"Vulkan",
    "gpu":gpu.stdout.strip(),
    "smoke_seconds":round(smoke_elapsed,1),
    "elapsed_seconds":round(elapsed,1),
    "elapsed_minutes":round(elapsed/60,2),
    "gpu_util_avg_percent":round(sum(utils)/len(utils),1) if utils else None,
    "gpu_util_max_percent":round(max(utils),1) if utils else None,
    "completed_at":datetime.now(timezone.utc).isoformat()
}
(out_dir/"colab-benchmark.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
print(json.dumps(summary,indent=2))

section("9/9 - Copiando resultados a Google Drive")
drive_project=cfg.get("driveProjectDir",PROJECT_SLUG)
drive_out=DRIVE_ROOT/drive_project/"renders-colab"
drive_out.mkdir(parents=True,exist_ok=True)
for f in out_dir.iterdir():
    if f.is_file():
        shutil.copy2(f,drive_out/f.name)

print("\nLISTO")
print("Destino:",drive_out)
for f in sorted(drive_out.iterdir()):
    if f.is_file():
        print(f" - {f.name}: {f.stat().st_size/1024/1024:.2f} MB")
